# LangExtract

Para rodar localmente e sem custos, baixe um modelo local (e.g. Llama3) -> https://ollama.com/download

Para testar o servidor, digite no terminal:

`ollama run llama3 "say hello"`

Para conferir se o servidor está conectado corretamente:

`ollama serve`

Para rodar localmente:

`ollama run gemma2:2b`

## First Test

In [1]:
import langextract as lx
import textwrap

# 1. Define a concise prompt
prompt = textwrap.dedent("""\
Extract structured product information from the text.
Identify product name, brand, model, category, color, size, material, and any key attributes.
Use the exact text for extractions — do not paraphrase.
Return relevant attributes that describe each product clearly.
""")

# 2. Provide a high-quality example to guide the model
examples = [
    lx.data.ExampleData(
        text="Camiseta PoloTech masculina de algodão, cor azul marinho, disponível nos tamanhos M, G e GG.",
        extractions=[
            lx.data.Extraction(
                extraction_class="product",
                extraction_text="Camiseta PoloTech masculina",
                attributes={
                    "brand": "PoloTech",
                    "category": "camiseta",
                    "material": "algodão",
                    "color": "azul marinho",
                    "sizes": ["M", "G", "GG"]
                },
            ),
        ],
    ),
    lx.data.ExampleData(
        text="Tênis esportivo Nike Air Zoom branco, ideal para corrida.",
        extractions=[
            lx.data.Extraction(
                extraction_class="product",
                extraction_text="Tênis esportivo Nike Air Zoom branco",
                attributes={
                    "brand": "Nike",
                    "category": "tênis esportivo",
                    "color": "branco",
                    "intended_use": "corrida"
                },
            ),
        ],
    ),
]

# 3. Run the extraction on your input text
input_text = "Bolsa feminina de couro sintético da marca Vizzano, cor bege, com alça ajustável e fechamento magnético."

result = lx.extract(
    text_or_documents=input_text,
    prompt_description=prompt,
    examples=examples,
    model_id="gemma2:2b",  # Automatically selects Ollama provider
    model_url="http://localhost:11434",
    fence_output=False,
    use_schema_constraints=False
)

for e in result.extractions:
    print(f"{e.extraction_class}: {e.extraction_text}")
    if e.attributes:
        print(f"Atributes: {e.attributes}")

LangExtract: Processing [00:09]

product: Bolsa feminina
Atributes: {'brand': 'Vizzano', 'category': 'bolsa feminina', 'color': 'bege', 'features': ['alça ajustável', 'fechamento magnético']}


# AE-110K Dataset

In [2]:
from datasets import load_dataset

dataset = load_dataset("av-generation/ae-110k-dataset")
dataset

/home/offerwise/Documents/JSONLLM/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['id', 'text', 'attributes', 'values', 'values_indices', 'values_text', 'attributes_values', 'json_answer', 'candidate_attributes', 'candidate_text', 'candidate_example'],
        num_rows: 31604
    })
    validation: Dataset({
        features: ['id', 'text', 'attributes', 'values', 'values_indices', 'values_text', 'attributes_values', 'json_answer', 'candidate_attributes', 'candidate_text', 'candidate_example'],
        num_rows: 3950
    })
    test: Dataset({
        features: ['id', 'text', 'attributes', 'values', 'values_indices', 'values_text', 'attributes_values', 'json_answer', 'candidate_attributes', 'candidate_text', 'candidate_example'],
        num_rows: 3951
    })
})

In [2]:
dataset['train'].features
dataset['train'][0]

{'id': 33651,
 'text': '48V Ebike battery 750W 48V 12AH Lithium Battery 48V 13S Electric Bike battery 48V 12AH with PVC case 20A BMS 54.6V 2A charger',
 'attributes': ['Capacity', 'Type', 'BMS'],
 'values': ['12AH', 'Lithium Battery', '20A'],
 'values_indices': [[27, 31], [32, 47], [101, 104]],
 'values_text': '12AH | Lithium Battery | 20A',
 'attributes_values': 'attribute: Capacity, value: 12AH | attribute: Type, value: Lithium Battery | attribute: BMS, value: 20A',
 'json_answer': "{'Capacity': '12AH', 'Type': 'Lithium Battery', 'BMS': '20A'}",
 'candidate_attributes': [''],
 'candidate_text': '',
 'candidate_example': {'json_answer': '', 'text': ''}}

In [3]:
texts = dataset['train']['text']
len(texts)

31604

In [ ]:
import json
import langextract as lx
import textwrap
from concurrent.futures import ThreadPoolExecutor

prompt = textwrap.dedent("""\
Extract structured product information from the text.
Identify product name, brand, model, category, color, size, material, and any key attributes.
Use the exact text for extractions — do not paraphrase.
Return relevant attributes that describe each product clearly.
""")

examples = [
    lx.data.ExampleData(
        text="Camiseta PoloTech masculina de algodão, cor azul marinho, disponível nos tamanhos M, G e GG.",
        extractions=[
            lx.data.Extraction(
                extraction_class="product",
                extraction_text="Camiseta PoloTech masculina",
                attributes={
                    "brand": "PoloTech",
                    "category": "camiseta",
                    "material": "algodão",
                    "color": "azul marinho",
                    "sizes": ["M", "G", "GG"]
                },
            ),
        ],
    ),
    lx.data.ExampleData(
        text="Tênis esportivo Nike Air Zoom branco, ideal para corrida.",
        extractions=[
            lx.data.Extraction(
                extraction_class="product",
                extraction_text="Tênis esportivo Nike Air Zoom branco",
                attributes={
                    "brand": "Nike",
                    "category": "tênis esportivo",
                    "color": "branco",
                    "intended_use": "corrida"
                },
            ),
        ],
    ),
]

output_path = "ae110k_extractions.jsonl"
MAX_WORKERS = 10


def extract_text(i, text):

    try:
        result = lx.extract(
            text_or_documents=text,
            prompt_description=prompt,
            examples=examples,
            model_id="gemma2:2b",
            model_url="http://localhost:11434",
            fence_output=False,
            use_schema_constraints=False,
            language_model_params={"timeout": 900}
        )

        if not result.extractions:
            return i, None

        e = result.extractions[0]
        attrs = e.attributes or {}

        # Montar campos no mesmo formato do AE-110K
        attributes = list(attrs.keys())
        values = list(attrs.values())
        values_text = " | ".join(map(str, values))
        attributes_values = " | ".join(
            f"attribute: {k}, value: {v}" for k, v in attrs.items()
        )
        json_answer = str(attrs)  # igual ao dataset (aspas simples)
        values_indices = []  

        record = {
            "id": i,
            "text": text,
            "attributes": attributes,
            "values": values,
            "values_indices": values_indices,
            "values_text": values_text,
            "attributes_values": attributes_values,
            "json_answer": json_answer,
        }

        return i, record

    except Exception as err:
        return i, {"error": str(err)}


with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor, open(output_path, "w") as f:
    for i, record in executor.map(lambda args: extract_text(*args), enumerate(texts)):
        if record and "error" not in record:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
            print(f"[{i}] OK")
        else:
            print(f"[{i}] ERROR: {record}")

LangExtract: Processing [00:00]



































LangExtract: Processing [00:32]
LangExtract: Processing [01:09]

LangExtract: Processing [01:40]



LangExtract: Processing [02:27]


[0] OK
[1] OK
[2] OK
[3] OK








LangExtract: Processing [03:11]










LangExtract: Processing [03:45]









LangExtract: Processing [04:21]


[4] OK










LangExtract: Processing [04:55]












LangExtract: Processing [05:22]


...













LangExtract: Processing [05:40]


[3438] OK

















LangExtract: Processing [05:47]


[3439] OK










LangExtract: Processing [05:44]


In [7]:
import json

converted = []

with open("ae110k_extractions.jsonl") as f:
    for line in f:
        obj = json.loads(line)
        attr_str = obj["attributes_values"]

        # divide cada par "attribute: X, value: Y"
        parts = [p.strip() for p in attr_str.split("|") if "attribute:" in p]

        pairs = {}
        for p in parts:
            if "attribute:" in p and "value:" in p:
                a = p.split("attribute:")[1].split(",")[0].strip()
                v = p.split("value:")[1].strip()
                if a and v and v.lower() != "none":
                    pairs[a] = v

        json_answer = "{" + ", ".join([f"'{k}': '{v}'" for k, v in pairs.items()]) + "}"

        converted.append({
            #"id": obj["id"],
            #"text": obj["text"],
            "json_answer": json_answer
        })

with open("ae110k_json_structured.jsonl", "w", encoding="utf-8") as f:
    for item in converted:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

# Evaluation

In [5]:
import json
from datasets import load_dataset
from evaluation import evaluate_solution
import numpy as np
import ast

In [6]:
dataset = load_dataset("av-generation/ae-110k-dataset")

predictions = [json.loads(line) for line in open("ae110k_extractions.jsonl")]

print(f"Predições: {len(predictions)} | Ground truth: {len(dataset['train'])}")

Predições: 3418 | Ground truth: 31604


In [19]:
dataset_slice = dataset["train"].select(range(len(predictions)))

results = []
for i, (pred_obj, gt_obj) in enumerate(zip(predictions, dataset_slice)):

    gt_json_str = gt_obj["json_answer"]
    if isinstance(gt_json_str, str):
        try:
            gt_json = json.loads(gt_json_str)
        except json.JSONDecodeError:
            gt_json = ast.literal_eval(gt_json_str)
    else:
        gt_json = gt_json_str
    gt_json = {k.lower(): v for k, v in gt_json.items() if v}

    model_json = pred_obj["attributes"]
    model_json = {k.lower(): v for k, v in model_json.items() if v}

    scores = evaluate_solution(model_json, gt_json)
    results.append(scores)

    if i % 100 == 0:
        print(f"[{i}] F1={scores['f1']} | P={scores['precision']} | R={scores['recall']}")

precision_mean = np.mean([r["precision"] for r in results])
recall_mean = np.mean([r["recall"] for r in results])
f1_mean = np.mean([r["f1"] for r in results])

print(f"Precision média: {precision_mean}")
print(f"Recall média: {recall_mean}")
print(f"F1 média: {f1_mean}")

[0] F1=0.0 | P=0.0 | R=0.0
[100] F1=0.0 | P=0.0 | R=0.0
[200] F1=0.0 | P=0.0 | R=0.0
[300] F1=0.0 | P=0.0 | R=0.0
[400] F1=0.0 | P=0.0 | R=0.0
[500] F1=0.0 | P=0.0 | R=0.0
[600] F1=0.0 | P=0.0 | R=0.0
[700] F1=0.0 | P=0.0 | R=0.0
[800] F1=0.0 | P=0.0 | R=0.0
[900] F1=0.0 | P=0.0 | R=0.0
[1000] F1=0.0 | P=0.0 | R=0.0
[1100] F1=0.0 | P=0.0 | R=0.0
[1200] F1=0.0 | P=0.0 | R=0.0
[1300] F1=0.0 | P=0.0 | R=0.0
[1400] F1=0.0 | P=0.0 | R=0.0
[1500] F1=0.0 | P=0.0 | R=0.0
[1600] F1=0.0 | P=0.0 | R=0.0
[1700] F1=0.0 | P=0.0 | R=0.0
[1800] F1=0.0 | P=0.0 | R=0.0
[1900] F1=0.0 | P=0.0 | R=0.0
[2000] F1=0.0 | P=0.0 | R=0.0
[2100] F1=0.0 | P=0.0 | R=0.0
[2200] F1=0.0 | P=0.0 | R=0.0
[2300] F1=0.0 | P=0.0 | R=0.0
[2400] F1=0.0 | P=0.0 | R=0.0
[2500] F1=0.0 | P=0.0 | R=0.0
[2600] F1=0.0 | P=0.0 | R=0.0
[2700] F1=0.0 | P=0.0 | R=0.0
[2800] F1=0.0 | P=0.0 | R=0.0
[2900] F1=0.0 | P=0.0 | R=0.0
[3000] F1=0.0 | P=0.0 | R=0.0
[3100] F1=0.0 | P=0.0 | R=0.0
[3200] F1=0.0 | P=0.0 | R=0.0
Precision média: 0.000

In [ ]:
import csv, ast

with open("comparison.csv", "w", newline='') as f:
    writer = csv.writer(f)
    writer.writerow(["index", "pred_key", "pred_value", "gt_key", "gt_value", "match"])

    for i, (pred_obj, gt_obj) in enumerate(zip(predictions, dataset_slice)):
        gt_json_str = gt_obj["json_answer"]
        gt_json = ast.literal_eval(gt_json_str) if isinstance(gt_json_str, str) else gt_json_str
        model_json = pred_obj["attributes"]

        all_keys = set(map(str.lower, list(model_json.keys()) + list(gt_json.keys())))
        for key in all_keys:
            pred_val = model_json.get(key) or model_json.get(key.lower())
            gt_val = gt_json.get(key) or gt_json.get(key.capitalize())
            match = (pred_val == gt_val)
            writer.writerow([i, key, pred_val, key, gt_val, match])


In [ ]:
with open("evaluation_summary.json", "w") as f:
    json.dump({
        "precision_mean": precision_mean,
        "recall_mean": recall_mean,
        "f1_mean": f1_mean,
        "n_pred": len(predictions),
        "n_gt": len(dataset["train"])
    }, f, indent=2, ensure_ascii=False)